### Catastrophe History — analytics dataset

Generates a large, realistic **historical** dataset for the catastrophe demo and
stores it as Delta tables in Unity Catalog (`{CATALOG}.orders`), for analytics
that the app / pipeline / runbooks consume later.

Fully Spark-native and deterministic (seeded by `CATASTROPHE_SEED`), so it scales
to millions of rows cheaply. Bakes in day-of-week + hour-of-day demand curves, a
mild growth trend, per-city volume, item/vehicle mix, and **bridge-outage
disruption** (a few closed-bridge days per city that spike delays, reroutes,
cancellations, refunds and complaints for river-crossing orders).

Tables written (all `overwrite`, idempotent) — bronze layer of the orders medallion:
- `bronze_hist_orders` — one row per order (the fact table)
- `bronze_hist_order_events` — status transitions (placed → enroute → … → delivered/cancelled)
- `bronze_hist_refunds` — refunds issued
- `bronze_hist_complaints` — customer complaints + resolutions
- `bronze_hist_actions` — operator actions (reroute, issue_credit, contact_customer, escalate)

In [ ]:
import sys
from datetime import date, timedelta

CATALOG = dbutils.widgets.get("CATALOG")


def _w(name, default):
    try:
        v = dbutils.widgets.get(name)
        return v if v not in (None, "") else default
    except Exception:
        return default


SIMULATOR_SCHEMA = _w("SIMULATOR_SCHEMA", "metadata")
# Historical bronze facts live in the orders medallion schema (not metadata).
ORDERS_SCHEMA = _w("ORDERS_SCHEMA", "orders")
SEED = int(_w("CATASTROPHE_SEED", "2026"))
SCENARIO_ID = _w("CATASTROPHE_SCENARIO", "bridge_outage")
TARGET_ORDERS = int(_w("HISTORY_TARGET_ORDERS", "2000000"))
DAYS = int(_w("HISTORY_DAYS", "90"))

print(
    f"catalog={CATALOG} orders_schema={ORDERS_SCHEMA} seed={SEED} "
    f"scenario={SCENARIO_ID} target_orders={TARGET_ORDERS} days={DAYS}"
)

In [ ]:
import pyspark.sql.functions as F

sys.path.append("../data/canonical")
from catastrophe_scenarios import (
    CITIES,
    SCENARIOS,
    amsterdam_kitchen_rows,
    generate_city_kitchens,
)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{ORDERS_SCHEMA}")

# Kitchens dimension (city_id + per-city index) built from the SAME registry the
# live app uses, so history rows join cleanly to the current kitchen list.
kitchen_rows = []
for cid in CITIES:
    ks = amsterdam_kitchen_rows() if cid == "amsterdam" else generate_city_kitchens(cid, seed=SEED)
    for idx, k in enumerate(ks):
        kitchen_rows.append(
            {
                "city_id": cid,
                "k_index": idx,
                "kitchen_id": k["kitchen_id"],
                "kitchen_name": k["name"],
                "neighborhood": k.get("neighborhood", ""),
                "lat": float(k["lat"]),
                "lon": float(k["lon"]),
            }
        )

kitchens_sdf = spark.createDataFrame(kitchen_rows)
city_counts = kitchens_sdf.groupBy("city_id").agg(F.count("*").alias("n_kitchens"))
print(f"kitchens dimension: {kitchens_sdf.count()} rows across {len(CITIES)} cities")

In [ ]:
# ── Demand grid: city × day × hour, weighted, then turned into exact counts ───
# Per-city relative order volume (bigger metros order more).
CITY_VOLUME = {
    "amsterdam": 1.0, "montreal": 1.1, "sao_paulo": 1.8, "vienna": 0.9,
    "warsaw": 1.1, "paris": 1.7, "lisbon": 1.0, "washington_dc": 1.3,
    "boston": 1.2, "bangalore": 1.9, "seoul": 1.8, "tokyo": 2.0,
    "chicago": 1.5, "minneapolis": 0.9,
}
cities_sdf = spark.createDataFrame(
    [(cid, CITIES[cid].name, float(CITY_VOLUME.get(cid, 1.0))) for cid in CITIES],
    ["city_id", "city_name", "city_w"],
)

# Days dimension: weekend boost + a mild growth trend over the window.
today = date.today()
start = today - timedelta(days=DAYS - 1)
day_rows = []
for d in range(DAYS):
    dt = start + timedelta(days=d)
    dow = dt.isoweekday()  # 1 Mon .. 7 Sun
    dow_w = {5: 1.25, 6: 1.45, 7: 1.30}.get(dow, 1.0)  # Fri/Sat/Sun busier
    trend = 1.0 + 0.35 * (d / max(1, DAYS - 1))
    day_rows.append((d, dt.isoformat(), dow, float(dow_w), float(trend)))
days_sdf = spark.createDataFrame(
    day_rows, ["day_index", "order_date", "dow", "dow_w", "trend"]
).withColumn("order_date", F.to_date("order_date"))

# Hour-of-day demand curve (breakfast / lunch / dinner peaks).
HOD_W = [0.15, 0.10, 0.08, 0.06, 0.06, 0.12, 0.30, 0.60, 0.80, 0.70, 0.60, 1.10,
         1.60, 1.50, 1.00, 0.70, 0.70, 1.10, 1.70, 2.00, 1.90, 1.50, 1.00, 0.50]
hours_sdf = spark.createDataFrame(
    [(h, float(HOD_W[h])) for h in range(24)], ["hour", "hod_w"]
)

grid = cities_sdf.crossJoin(days_sdf).crossJoin(hours_sdf)
# Deterministic per-cell noise from the seed so counts are reproducible.
noise = 0.85 + (F.pmod(F.hash("city_id", "day_index", "hour", F.lit(SEED)), F.lit(300)) / F.lit(1000.0))
grid = grid.withColumn("cell_w", F.col("city_w") * F.col("dow_w") * F.col("hod_w") * F.col("trend") * noise)

sum_w = grid.agg(F.sum("cell_w").alias("s")).collect()[0]["s"]
grid = grid.withColumn(
    "n", F.round(F.col("cell_w") / F.lit(float(sum_w)) * F.lit(TARGET_ORDERS)).cast("int")
).where(F.col("n") > 0)
print(f"grid cells producing orders: {grid.count()}")

In [ ]:
# ── Explode grid cells into individual orders, assign attributes ──────────────
def u(salt):
    """Deterministic uniform [0,1) per order + salt (reproducible, no rand())."""
    return F.pmod(F.hash("order_id", F.lit(salt)), F.lit(1_000_000)) / F.lit(1_000_000.0)


orders = (
    grid.withColumn("seq", F.explode(F.sequence(F.lit(1), F.col("n"))))
    .withColumn(
        "order_id",
        F.concat_ws(
            "-",
            "city_id",
            F.lpad(F.col("day_index").cast("string"), 3, "0"),
            F.lpad(F.col("hour").cast("string"), 2, "0"),
            F.lpad(F.col("seq").cast("string"), 4, "0"),
        ),
    )
)

# placed timestamp: within the cell's hour, deterministic sub-hour offset.
start_ts = F.to_timestamp(F.lit(start.isoformat()))
orders = orders.withColumn(
    "order_ts",
    (
        F.unix_timestamp(start_ts)
        + F.col("day_index") * 86400
        + F.col("hour") * 3600
        + F.pmod(F.hash("order_id"), F.lit(3600))
    ).cast("timestamp"),
)

# kitchen: uniform pick within the city's kitchen list.
orders = orders.join(F.broadcast(city_counts), "city_id", "left")
orders = orders.withColumn("k_index", (u("kitchen") * F.col("n_kitchens")).cast("int"))
orders = orders.join(F.broadcast(kitchens_sdf), ["city_id", "k_index"], "left")

# vehicle (weighted): bike .45 / car .30 / scooter .25
uv = u("veh")
orders = orders.withColumn(
    "vehicle", F.when(uv < 0.45, "bike").when(uv < 0.75, "car").otherwise("scooter")
)

# kind (weights hot 5 / grocery 2 / ice 3 / frozen 2 = 12) — matches app ORDER_KINDS.
uk = u("kind")
orders = orders.withColumn(
    "kind",
    F.when(uk < 5 / 12, "hot")
    .when(uk < 7 / 12, "grocery")
    .when(uk < 10 / 12, "ice")
    .otherwise("frozen"),
)
orders = orders.withColumn(
    "kind_label",
    F.when(F.col("kind") == "hot", "Hot food")
    .when(F.col("kind") == "grocery", "Groceries")
    .when(F.col("kind") == "ice", "Ice cream")
    .otherwise("Frozen"),
)
orders = orders.withColumn("cold", F.col("kind").isin("ice", "frozen"))
# per-kind max tolerated delay (min past promise) before auto-cancel — app maxima.
orders = orders.withColumn(
    "max_delay",
    F.when(F.col("kind") == "hot", 25)
    .when(F.col("kind") == "grocery", 40)
    .when(F.col("kind") == "ice", 15)
    .otherwise(20),
)

orders = orders.withColumn("cross_river", u("cross") < 0.60)

# bridge closure ~3% of days per city (≈ 2-3 outage days over a 90-day window).
orders = orders.withColumn(
    "bridge_closed",
    F.pmod(F.hash("city_id", "day_index", F.lit(SEED), F.lit("outage")), F.lit(100)) < 3,
)
orders = orders.withColumn("scenario_id", F.lit(SCENARIO_ID))
orders = orders.withColumn("disrupted", F.col("bridge_closed") & F.col("cross_river"))

In [ ]:
# ── Timings, disruption, outcome (delivered vs cancelled), refunds, complaints ─
DELAY_MULT = (
    float(SCENARIOS[SCENARIO_ID].delivery_delay_multiplier)
    if SCENARIO_ID in SCENARIOS
    else 2.0
)

orders = (
    orders.withColumn("prep_min", 8 + u("prep") * 12)
    .withColumn(
        "travel_base",
        F.when(F.col("vehicle") == "bike", 18.0)
        .when(F.col("vehicle") == "car", 12.0)
        .otherwise(14.0),
    )
    .withColumn("travel_min", F.col("travel_base") + u("travel") * 10)
    .withColumn("eta_min", F.col("prep_min") + F.col("travel_min") + F.lit(5.0))
)

# Base delay: heavy-tailed but usually small. Catastrophe adds a large delay to
# river-crossing orders on closed-bridge days; non-crossing orders get a little
# spillover congestion.
orders = (
    orders.withColumn("base_extra", F.pow(u("delay"), F.lit(3.0)) * 22.0)
    .withColumn(
        "spill_extra",
        F.when(F.col("bridge_closed"), u("spill") * 15.0).otherwise(F.lit(0.0)),
    )
    .withColumn(
        "cat_extra",
        F.when(F.col("disrupted"), (30.0 + u("cat") * 150.0) * F.lit(DELAY_MULT)).otherwise(
            F.lit(0.0)
        ),
    )
)
# Reroute: disrupted orders are often rerouted, which cuts (but doesn't erase)
# the catastrophe delay.
orders = orders.withColumn("rerouted", F.col("disrupted") & (u("reroute") < 0.55))
orders = orders.withColumn(
    "cat_extra",
    F.when(F.col("rerouted"), F.col("cat_extra") * 0.55 + 12.0).otherwise(F.col("cat_extra")),
)
orders = orders.withColumn(
    "late_min",
    F.greatest(
        F.lit(0),
        F.round(F.col("base_extra") + F.col("spill_extra") + F.col("cat_extra")).cast("int"),
    ),
)

# Cancel: over tolerance → likely cancel; otherwise a small base cancel rate.
ucn = u("cancel")
orders = orders.withColumn(
    "cancelled",
    F.when(F.col("late_min") > F.col("max_delay"), ucn < 0.70).otherwise(ucn < 0.02),
)
orders = orders.withColumn(
    "status", F.when(F.col("cancelled"), "cancelled").otherwise("delivered")
)

# promise / completion timestamps
orders = orders.withColumn(
    "promised_ts",
    (F.unix_timestamp("order_ts") + (F.col("eta_min") * 60).cast("long")).cast("timestamp"),
)
orders = orders.withColumn(
    "delivered_ts",
    F.when(
        F.col("status") == "delivered",
        (F.unix_timestamp("promised_ts") + (F.col("late_min") * 60).cast("long")).cast("timestamp"),
    ),
)
orders = orders.withColumn(
    "cancelled_ts",
    F.when(
        F.col("status") == "cancelled",
        (F.unix_timestamp("promised_ts") + (F.col("max_delay") * 60).cast("long")).cast("timestamp"),
    ),
)

# order value by kind
orders = orders.withColumn(
    "order_value",
    F.round(
        F.when(F.col("kind") == "hot", 18 + u("val") * 27)
        .when(F.col("kind") == "grocery", 25 + u("val") * 55)
        .when(F.col("kind") == "ice", 8 + u("val") * 12)
        .otherwise(15 + u("val") * 25),
        2,
    ),
)

# refund + complaint flags
orders = orders.withColumn(
    "refunded",
    F.when(F.col("cancelled"), u("refund") < 0.85)
    .when(
        F.col("late_min") > (F.col("max_delay") * 0.6),
        u("refund") < F.when(F.col("cold"), 0.35).otherwise(0.22),
    )
    .otherwise(F.lit(False)),
)
orders = orders.withColumn(
    "complaint",
    F.when(F.col("cancelled"), u("compl") < F.when(F.col("cold"), 0.62).otherwise(0.45))
    .when(F.col("late_min") > F.col("max_delay"), u("compl") < 0.30)
    .otherwise(F.lit(False)),
)

In [ ]:
# ── Write the orders fact table ──────────────────────────────────────────────
fact = orders.select(
    "order_id",
    "order_date",
    "order_ts",
    "promised_ts",
    "delivered_ts",
    "cancelled_ts",
    "city_id",
    "city_name",
    "kitchen_id",
    "kitchen_name",
    "neighborhood",
    F.col("lat").alias("kitchen_lat"),
    F.col("lon").alias("kitchen_lon"),
    "vehicle",
    "kind",
    "kind_label",
    "cold",
    "cross_river",
    "scenario_id",
    "bridge_closed",
    "disrupted",
    "rerouted",
    "status",
    "late_min",
    F.round("eta_min", 1).alias("eta_min"),
    F.round("prep_min", 1).alias("prep_min"),
    F.round("travel_min", 1).alias("travel_min"),
    "order_value",
    "refunded",
    "complaint",
    "dow",
    "hour",
)

ORDERS_TBL = f"{CATALOG}.{ORDERS_SCHEMA}.bronze_hist_orders"
(
    fact.write.mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("city_id")
    .saveAsTable(ORDERS_TBL)
)
n_orders = spark.table(ORDERS_TBL).count()
print(f"{ORDERS_TBL}: {n_orders:,} rows")

In [ ]:
# ── Derive child tables from the persisted fact (guarantees consistency) ──────
f = spark.table(ORDERS_TBL)

# Status timeline: placed → enroute → [rerouted] → [stuck] → delivered/cancelled
enroute_ts = (F.unix_timestamp("order_ts") + (F.col("prep_min") * 60).cast("long")).cast("timestamp")
reroute_ts = (F.unix_timestamp("order_ts") + (F.col("prep_min") * 60 + 120).cast("long")).cast("timestamp")
final_ts = F.coalesce("delivered_ts", "cancelled_ts")

timeline = F.array(
    F.struct(F.lit("placed").alias("status"), F.col("order_ts").alias("at"), F.lit(0).alias("late_min")),
    F.struct(F.lit("enroute").alias("status"), enroute_ts.alias("at"), F.lit(0).alias("late_min")),
    F.when(
        F.col("rerouted"),
        F.struct(F.lit("rerouted").alias("status"), reroute_ts.alias("at"), F.lit(0).alias("late_min")),
    ),
    F.when(
        F.col("late_min") >= 5,
        F.struct(F.lit("stuck").alias("status"), F.col("promised_ts").alias("at"), F.col("late_min").alias("late_min")),
    ),
    F.struct(F.col("status").alias("status"), final_ts.alias("at"), F.col("late_min").alias("late_min")),
)

events = (
    f.select("order_id", "city_id", "order_ts", "prep_min", "promised_ts", "delivered_ts", "cancelled_ts", "rerouted", "late_min", "status")
    .withColumn("tl", F.filter(timeline, lambda x: x.isNotNull()))
    .withColumn("e", F.explode("tl"))
    .select(
        "order_id",
        "city_id",
        F.col("e.status").alias("status"),
        F.col("e.at").alias("event_ts"),
        F.col("e.late_min").alias("late_min"),
    )
)

EVENTS_TBL = f"{CATALOG}.{ORDERS_SCHEMA}.bronze_hist_order_events"
events.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(EVENTS_TBL)
print(f"{EVENTS_TBL}: {spark.table(EVENTS_TBL).count():,} rows")

In [ ]:
# ── Refunds ──────────────────────────────────────────────────────────────────
REFUND_REASONS = [
    "Late delivery",
    "Cold item at risk",
    "Customer goodwill credit",
    "Partial order issue",
]
_reason_arr = F.array(*[F.lit(r) for r in REFUND_REASONS])
_reason_idx = F.pmod(F.hash("order_id", F.lit("rreason")), F.lit(len(REFUND_REASONS)))

refunds = (
    f.where("refunded")
    .withColumn(
        "reason",
        F.when(F.col("status") == "cancelled", F.lit("Order cancelled"))
        .when(F.col("disrupted"), F.lit("Bridge outage disruption"))
        .otherwise(F.element_at(_reason_arr, _reason_idx + 1)),
    )
    .withColumn(
        "refund_amount",
        F.round(F.col("order_value") * F.when(F.col("status") == "cancelled", 1.0).otherwise(0.5), 2),
    )
    .withColumn("issued_at", F.coalesce("cancelled_ts", "delivered_ts", "promised_ts"))
    .select("order_id", "city_id", "kind", "reason", "order_value", "refund_amount", "issued_at")
)

REFUNDS_TBL = f"{CATALOG}.{ORDERS_SCHEMA}.bronze_hist_refunds"
refunds.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(REFUNDS_TBL)
print(f"{REFUNDS_TBL}: {spark.table(REFUNDS_TBL).count():,} rows")

In [ ]:
# ── Complaints ───────────────────────────────────────────────────────────────
QUOTES = [
    "Where is my order?! No one told me it was cancelled.",
    "This is unacceptable, I've been waiting forever.",
    "You cancelled my order and didn't even refund me?",
    "My ice cream would have melted by now.",
    "Worst delivery experience ever.",
    "I want a refund and an apology.",
]
_quote_arr = F.array(*[F.lit(q) for q in QUOTES])
_quote_idx = F.pmod(F.hash("order_id", F.lit("quote")), F.lit(len(QUOTES)))
_resolved = F.pmod(F.hash("order_id", F.lit("resolve")), F.lit(100)) < 65

complaints = (
    f.where("complaint")
    .withColumn("quote", F.element_at(_quote_arr, _quote_idx + 1))
    .withColumn("sentiment", F.when(F.col("cold"), "very_negative").otherwise("negative"))
    .withColumn(
        "resolution",
        F.when(_resolved, F.when(F.col("refunded"), "apology_refund").otherwise("apology_credit")),
    )
    .withColumn("raised_at", F.coalesce("cancelled_ts", "delivered_ts", "promised_ts"))
    .withColumn(
        "resolved_at",
        F.when(
            _resolved,
            (
                F.unix_timestamp(F.coalesce("cancelled_ts", "delivered_ts", "promised_ts"))
                + (F.pmod(F.hash("order_id", F.lit("rt")), F.lit(120)) + 10) * 60
            ).cast("timestamp"),
        ),
    )
    .select("order_id", "city_id", "kind", "cold", "quote", "sentiment", "resolution", "raised_at", "resolved_at")
)

COMPLAINTS_TBL = f"{CATALOG}.{ORDERS_SCHEMA}.bronze_hist_complaints"
complaints.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(COMPLAINTS_TBL)
print(f"{COMPLAINTS_TBL}: {spark.table(COMPLAINTS_TBL).count():,} rows")

In [ ]:
# ── Operator actions (reroute / issue_credit / contact_customer / escalate) ───
def _action(cond, atype, notes_col, ts_col):
    return f.where(cond).select(
        F.concat_ws("-", F.lit(atype), "order_id").alias("action_id"),
        F.lit(atype).alias("action_type"),
        "order_id",
        "city_id",
        notes_col.alias("notes"),
        ts_col.alias("created_at"),
    )


reroute_actions = _action(
    F.col("rerouted"), "reroute",
    F.lit("Driver rerouted via alternate crossing"), F.col("promised_ts"),
)
credit_actions = _action(
    F.col("refunded"), "issue_credit",
    F.concat(F.lit("Refund issued: $"), F.col("order_value").cast("string")),
    F.coalesce("cancelled_ts", "delivered_ts", "promised_ts"),
)
contact_actions = _action(
    F.col("complaint"), "contact_customer",
    F.lit("Support contacted the customer"),
    F.coalesce("cancelled_ts", "delivered_ts", "promised_ts"),
)
escalate_actions = _action(
    F.col("complaint") & (F.pmod(F.hash("order_id", F.lit("esc")), F.lit(100)) < 25),
    "escalate",
    F.lit("Escalated to on-call ops manager"),
    F.coalesce("cancelled_ts", "delivered_ts", "promised_ts"),
)

actions = (
    reroute_actions.unionByName(credit_actions)
    .unionByName(contact_actions)
    .unionByName(escalate_actions)
)

ACTIONS_TBL = f"{CATALOG}.{ORDERS_SCHEMA}.bronze_hist_actions"
actions.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(ACTIONS_TBL)
print(f"{ACTIONS_TBL}: {spark.table(ACTIONS_TBL).count():,} rows")

In [ ]:
# ── Summary ──────────────────────────────────────────────────────────────────
print("Catastrophe history generated:")
for t in (ORDERS_TBL, EVENTS_TBL, REFUNDS_TBL, COMPLAINTS_TBL, ACTIONS_TBL):
    print(f"  {t}: {spark.table(t).count():,} rows")

print("\nBy status:")
spark.table(ORDERS_TBL).groupBy("status").count().orderBy(F.desc("count")).show(truncate=False)

print("Disruption impact (avg late_min, cancel rate by disrupted flag):")
(
    spark.table(ORDERS_TBL)
    .groupBy("disrupted")
    .agg(
        F.count("*").alias("orders"),
        F.round(F.avg("late_min"), 1).alias("avg_late_min"),
        F.round(F.avg(F.when(F.col("status") == "cancelled", 1.0).otherwise(0.0)), 3).alias("cancel_rate"),
    )
    .orderBy("disrupted")
    .show(truncate=False)
)